Задача

In [2]:
import pandas as pd
import numpy as np

In [3]:

df = pd.read_csv("data/task_2_data_ex.csv")

In [57]:
fin_material = df[df['produced_material_release_type'] == 'FIN']

In [27]:
cleandf = df[['plant_id','year','produced_material','produced_material_release_type','component_material_production_type','component_material','component_material_release_type']]

In [72]:
cleandf[cleandf['component_material_release_type'] == 'RM']

,plant_id,year,produced_material,produced_material_release_type,component_material_production_type,component_material,component_material_release_type
9,RLT_10,2024,80000,PROD,NaN,70000,RM
20,RLT_10,2024,80000,PROD,NaN,70000,RM
31,RLT_10,2024,80000,PROD,NaN,70000,RM
42,RLT_10,2024,80000,PROD,NaN,70000,RM
53,RLT_10,2024,80000,PROD,NaN,70000,RM
...,...,...,...,...,...,...,...
1274,RLT_14,2024,80009,PROD,NaN,70009,RM
1285,RLT_14,2024,80009,PROD,NaN,70009,RM
1296,RLT_14,2024,80009,PROD,NaN,70009,RM
1307,RLT_14,2024,80009,PROD,NaN,70009,RM


In [99]:
fin  = df[df['produced_material_release_type'] == 'FIN']
prod = df[df['produced_material_release_type'] != 'FIN']

layer1 = fin.merge(
    prod,
    left_on  = ['plant_id', 'year', 'month', 'component_material'],
    right_on = ['plant_id', 'year', 'month', 'produced_material'],
    suffixes = ('_fin', '_prod')
)

print(layer1[['produced_material_fin', 'produced_material_prod', 'component_material_prod']])

     produced_material_fin  produced_material_prod  component_material_prod
0                    10000                   50000                    80070
1                    10000                   50000                    90000
2                    10000                   50000                    90001
3                    10000                   50000                    80070
4                    10000                   50000                    90000
..                     ...                     ...                      ...
355                  10009                   50009                    90045
356                  10009                   50009                    90046
357                  10009                   50009                    80079
358                  10009                   50009                    90045
359                  10009                   50009                    90046

[360 rows x 3 columns]


In [4]:
#fin
#
master_lookup = df.copy()

# Layer 2: Finding the recipe for the components found in Layer 1
layer2 = layer1.merge(
    master_lookup,
    left_on  = ['plant_id', 'year', 'month', 'component_material_prod'], # The "child" from Layer 1
    right_on = ['plant_id', 'year', 'month', 'produced_material'],      # Find its "recipe"
    suffixes = ('', '_layer2') 
)

# Now you can see: FIN -> Stage 1 -> Stage 2
# The column 'component_material' comes from master_lookup
# It didn't get a suffix because 'component_material_prod' (left) != 'component_material' (right)
print(layer2[['produced_material_fin', 'produced_material_prod', 'component_material']])

     produced_material_fin  produced_material_prod  component_material
0                    10000                   50000               80010
1                    10003                   50003               80013
2                    10003                   50003               90017
3                    10000                   50000               90002
4                    10003                   50003               90018
..                     ...                     ...                 ...
355                  10009                   50009               90048
356                  10009                   50009               80019
357                  10003                   50003               90018
358                  10009                   50009               90047
359                  10009                   50009               90048

[360 rows x 3 columns]


In [123]:
# Select only the columns needed for the report
final_report = layer2[[
    'plant_id', 'year', 'produced_material', 
    'produced_material_release_type', 'produced_material_production_type', 
    'component_material'
]]

# Rename to the clean headers you specified
final_report.columns = ['Plant', 'Year', 'Material', 'Release_Type', 'Prod_Type', 'Component']

In [ ]:
# 1. Start with your FIN materials
current_level = fin.copy()
all_layers = []

while not current_level.empty:
    # Add the current batch of rows to our collection
    all_layers.append(current_level)
    
    # 2. Find the "Children": Search for the next step in the 'prod' dataframe
    # We match the 'component_material' of the current step 
    # to the 'produced_material' of the next step
    current_level = prod.merge(
        current_level[['plant_id', 'year', 'month', 'component_material']].drop_duplicates(),
        left_on=['plant_id', 'year', 'month', 'produced_material'],
        right_on=['plant_id', 'year', 'month', 'component_material']
    )
    
    # Clean up the merge columns so it's ready for the next loop
    current_level = current_level.drop(columns=['component_material_y']).rename(
        columns={'component_material_x': 'component_material'}
    )

# 3. Stack everything together
final_output = pd.concat(all_layers, ignore_index=True)

In [129]:
# Select only the columns needed for the report
final_report = final_output[[
    'plant_id', 'year', 'produced_material', 
    'produced_material_release_type', 'produced_material_production_type', 
    'component_material'
]]

# Rename to the clean headers you specified
final_report.columns = ['Plant', 'Year', 'Material', 'Release_Type', 'Prod_Type', 'Component']

In [130]:
print(final_report)

       Plant  Year  Material Release_Type  Prod_Type  Component
0     RLT_10  2024     10000          FIN       8002      50000
1     RLT_10  2024     10000          FIN       8002      50000
2     RLT_10  2024     10000          FIN       8002      50000
3     RLT_10  2024     10000          FIN       8002      50000
4     RLT_10  2024     10000          FIN       8002      50000
...      ...   ...       ...          ...        ...        ...
1315  RLT_14  2024     80009         PROD       8000      90050
1316  RLT_14  2024     80009         PROD       8000      70009
1317  RLT_14  2024     80009         PROD       8000      90050
1318  RLT_14  2024     80009         PROD       8000      70009
1319  RLT_14  2024     80009         PROD       8000      90050

[1320 rows x 6 columns]


In [131]:
df.to_csv('filename.csv', index=False)